### Lab 2.2: Perceptron Algorithm in PyTorch

In this lab you will again implement the perceptron algorithm, but this time using PyTorch.

In [1]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import torch

PyTorch is very similar to NumPy in its basic functionality.  In PyTorch arrays are called tensors.

In [3]:
a = torch.tensor(5)
a

tensor(5)

In [4]:
b = torch.tensor(6)
a+b

tensor(11)

In [5]:
c = torch.zeros(3,5).float()
c

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

*A note on broadcasting:* You may have noticed in the previous lab that NumPy is particular about the sizes of the arrays in operations; PyTorch is the same way.

For example, if `A` has shape `(10,5)` and `b` has shape `(10,)`, then we can't compute `A*b`.  It wants the *last* dimensions to match, not the first ones.  So you would need to do either `A.T*b`.

In [6]:
A = np.random.normal(size=(10,5))
b = np.ones(10)

In [7]:
try:
    A*b
except ValueError as e:
    print(e)

operands could not be broadcast together with shapes (10,5) (10,) 


In [8]:
A.T*b

array([[ 1.71526038,  0.68459223, -2.26563383, -0.51691387,  0.46817027,
        -0.40408177, -0.17175833,  0.54554188,  0.03907117, -0.36439376],
       [-1.97209857,  1.34339687, -1.9919792 , -0.29550371,  0.18438678,
         1.8991854 ,  1.34269779, -2.0969581 ,  1.33507895, -0.2616214 ],
       [-0.2344312 ,  0.37502532, -0.14133178, -1.02677116,  0.96612063,
         1.3081863 ,  1.02231952,  0.76858937, -0.12110409, -0.05134848],
       [ 1.62595528, -2.26908868,  0.58561095, -0.89060445, -0.44646544,
         1.75721104,  1.14165176, -2.45595334,  1.05919657,  0.60983753],
       [ 0.64927688, -2.00385991,  0.38133571,  0.01843973, -0.50603363,
         0.79337733,  0.02288301, -0.11906528, -1.60075454,  1.41719365]])

An alternative is to introduce an extra dimension of size one to $b$.  However, note that this produces the transposed result from before.

In [9]:
A*b[:,None]

array([[ 1.71526038, -1.97209857, -0.2344312 ,  1.62595528,  0.64927688],
       [ 0.68459223,  1.34339687,  0.37502532, -2.26908868, -2.00385991],
       [-2.26563383, -1.9919792 , -0.14133178,  0.58561095,  0.38133571],
       [-0.51691387, -0.29550371, -1.02677116, -0.89060445,  0.01843973],
       [ 0.46817027,  0.18438678,  0.96612063, -0.44646544, -0.50603363],
       [-0.40408177,  1.8991854 ,  1.3081863 ,  1.75721104,  0.79337733],
       [-0.17175833,  1.34269779,  1.02231952,  1.14165176,  0.02288301],
       [ 0.54554188, -2.0969581 ,  0.76858937, -2.45595334, -0.11906528],
       [ 0.03907117,  1.33507895, -0.12110409,  1.05919657, -1.60075454],
       [-0.36439376, -0.2616214 , -0.05134848,  0.60983753,  1.41719365]])

In [10]:
A*np.expand_dims(b,-1)

array([[ 1.71526038, -1.97209857, -0.2344312 ,  1.62595528,  0.64927688],
       [ 0.68459223,  1.34339687,  0.37502532, -2.26908868, -2.00385991],
       [-2.26563383, -1.9919792 , -0.14133178,  0.58561095,  0.38133571],
       [-0.51691387, -0.29550371, -1.02677116, -0.89060445,  0.01843973],
       [ 0.46817027,  0.18438678,  0.96612063, -0.44646544, -0.50603363],
       [-0.40408177,  1.8991854 ,  1.3081863 ,  1.75721104,  0.79337733],
       [-0.17175833,  1.34269779,  1.02231952,  1.14165176,  0.02288301],
       [ 0.54554188, -2.0969581 ,  0.76858937, -2.45595334, -0.11906528],
       [ 0.03907117,  1.33507895, -0.12110409,  1.05919657, -1.60075454],
       [-0.36439376, -0.2616214 , -0.05134848,  0.60983753,  1.41719365]])

In general, carefully check the sizes of all arrays in your code!

In [11]:
from palmerpenguins import load_penguins
from mlxtend.plotting import plot_decision_regions
from matplotlib import pyplot as plt

c:\Users\Logan\anaconda3\envs\csc487_env\Lib\site-packages\palmerpenguins\penguins.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Here we loading and format the Palmer penguins dataset for binary classification.

In [12]:
df = load_penguins()

# drop rows with missing values
df.dropna(inplace=True)

# tricky code to randomly shuffle the rows
df = df.sample(frac=1).reset_index(drop=True)

# select only two specices
df = df[(df['species']=='Adelie')|(df['species']=='Chinstrap')]

# get two features
X = df[['flipper_length_mm','bill_length_mm']].values

# convert speces labels to 0 and 1
y = df['species'].map({'Adelie':0,'Chinstrap':1}).values

To make the learning algorithm work more smoothly, we we will subtract the mean of each feature.

Here `np.mean` calculates a mean, and `axis=0` tells NumPy to calculate the mean over the rows (calculate the mean of each column).

In [13]:
X -= np.mean(X,axis=0)

Now we will convert our `X` and `y` arrays to torch Tensors.

In [14]:
X = torch.tensor(X).float()
y = torch.tensor(y).float()

In [15]:
print(X.shape)
print(y.shape)
print(y)

torch.Size([214, 2])
torch.Size([214])
tensor([1., 0., 0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 1., 0., 1., 0., 0., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 1., 0., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0., 1., 1., 1., 1., 0.,
        0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0.,
        0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 0.,
        0., 0., 0., 0., 1., 1., 1., 0., 0., 1., 0., 1., 0., 1., 0., 1., 0., 0.,
        0., 0., 0., 0., 1., 1., 0., 0., 1., 0., 0., 1., 0., 1., 1., 1., 0., 0.,
        0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1.,
        0., 0., 0., 1., 1., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0.,
        1., 1., 1., 0., 0., 0., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 1., 1.,
        1., 0., 1., 0., 0., 1., 0., 1., 1., 0., 1., 0., 1., 0., 0., 0.])


### Exercises

Your task is to again complete this class for the perceptron, with two changes from last time:
- the implementation should use PyTorch tensors, not NumPy arrays;
- `train_step` now accepts the entire dataset as input and should calculate the average gradient over all examples, rather than updating the weights one data point at a time.

In [16]:
class Perceptron:
    def __init__(self,lr=1e-3):
        # store the learning rate
        self.lr = lr

        # initialize the weights to small, normally-distributed values
        self.w = torch.normal(mean=0, std=0.01, size=(2,))

        # initialize the bias to zero
        self.b = torch.zeros(1)

    def train_step(self,X:torch.Tensor,y:torch.Tensor) -> None:
        """ Apply the first update rule shown in lecture.
            Arguments:
             X: data matrix of shape (N,2)
             y: labels of shape (N,) 
        """
        # WRITE CODE HERE
        n = X.shape[0]
        y_adjust = torch.where(y==0, -1, 1)

        w_comb: torch.Tensor = torch.cat([self.w, self.b])
        X_comb: torch.Tensor = torch.cat([X,torch.ones(n,1)], dim=1)

        # grad_L = torch.mean((w_comb@X_comb.T - y_adjust) @ X_comb)
        grad_L = ((w_comb@X_comb.T - y_adjust) @ X_comb) / n
        self.w = self.w - self.lr*grad_L[:-1]
        self.b = self.b - self.lr*grad_L[-1]
    
    def predict(self,X:torch.Tensor) -> torch.Tensor:
        """ Calculate model prediction for all data points.
            Arguments:
             X: data matrix of shape (N,2)   
            Returns:
             Predicted labels (0 or 1) of shape (N,)
        """
        # WRITE CODE HERE
        z = self.w@X.T + self.b
        return torch.where(z >= 0, 1, 0)
    
    def score(self,X:torch.Tensor,y:torch.Tensor) -> torch.Tensor:
        """ Calculate model accuracy
            Arguments:
             X: data matrix of shape (N,2)   
             y: labels of shape (N,)
            Returns:
             Accuracy score
        """
        # WRITE CODE HERE
        y_predict = self.predict(X)
        y_correct = y_predict == y
        return torch.sum(y_correct) / len(y_correct)
        


Run the following code to train the model and print out the accuracy at each step.

In [18]:
lr = 1e-2
epochs = 100
model = Perceptron(lr)
for i in range(epochs):
    model.train_step(X,y)
    print(f'step {i}: {model.score(X,y)}')

step 0: 0.8317757248878479
step 1: 0.8504672646522522
step 2: 0.8598130941390991
step 3: 0.8831775784492493
step 4: 0.8785046935081482
step 5: 0.8831775784492493
step 6: 0.9158878326416016
step 7: 0.9252336621284485
step 8: 0.9345794320106506
step 9: 0.9299065470695496
step 10: 0.9299065470695496
step 11: 0.9299065470695496
step 12: 0.9299065470695496
step 13: 0.9345794320106506
step 14: 0.9392523169517517
step 15: 0.9392523169517517
step 16: 0.9392523169517517
step 17: 0.9392523169517517
step 18: 0.9392523169517517
step 19: 0.9392523169517517
step 20: 0.9439252614974976
step 21: 0.9439252614974976
step 22: 0.9439252614974976
step 23: 0.9439252614974976
step 24: 0.9439252614974976
step 25: 0.9439252614974976
step 26: 0.9439252614974976
step 27: 0.9439252614974976
step 28: 0.9485981464385986
step 29: 0.9485981464385986
step 30: 0.9532710313796997
step 31: 0.9532710313796997
step 32: 0.9532710313796997
step 33: 0.9532710313796997
step 34: 0.9532710313796997
step 35: 0.9532710313796997
st

Run the training multiple times.  Is the training the same each time, or does it vary?  Why?

The training is slightly different each time because the starting weights are randomized each time a new Perceptron instance is created. This results in slight differences during the training, as the gradients will be different and thus the weights and resulting accuracy will be different throughout the timesteps.

Play with the learning rate and number of epochs to find the best setting.

With the default learning rate of 0.001 and 100 epochs, the resulting training accuracy was about 93% each time. Upping to 500 epochs raised the accuracy to about 95%. Setting 1500 epochs and beyond, it can be seen that the accuracy converges about just under 96%, which happens a few epochs after 1400.

Returning the number of epochs to 100, increasing the learning rate by a factor of 10 to 0.01 resulted in a training accuracy of 95-96%, the same accuracy converged to when the number of epochs was increased beyon 1400. However, increasing the learning rate by 10x again up to 0.1 (still with 100 epochs), the accuracy of the model oscillated between 25% and 75% for nearly 50 epochs, before finally settling at only 68% accuracy. Even increasing the epochs to 1500 with this high 0.1 learning rate, the final accuracy was still only 68%. Decreasing the learning rate down to 0.0001, it unsurprisingly took about 5000-6000 epochs for the model to converge at the best accuracy of ~96%.

Overall, it seems that in this case the best setting for efficient training is still 100 epochs but at a learning rate of about 0.01, rather than 0.001.
